In [ ]:
import ROOT as R
import itertools
import math
import csv
import os


data_files = "~/hps/track_cluster_matching/data/*.root"
df_data = R.RDataFrame("MiniDST", data_files)
print("Events in data:", df_data.Count().GetValue())

mc_files = "~/hps/ecal_calibration/data/fee_mc/*.root"
df_mc = R.RDataFrame("MiniDST", mc_files)
print("Events in MC:", df_mc.Count().GetValue())

Events in data: 1918784
Events in MC: 617450


In [4]:
# Plot mean or sum energy maps in x-y bins, for data and MC, side by side.

def mean_energy_seed_map(df, name, title,
                         nx=47, xmin=-23.5, xmax=23.5,
                         ny=11, ymin=-5.5, ymax=5.5):
    """
    Mean cluster energy per SEED crystal (ix,iy).
    Each bin = one ECAL crystal.
    """

    df2 = df.Define("clus_one",
                    "ROOT::VecOps::RVec<float>(ecal_cluster_energy.size(), 1.0f)")

    h_sumE = df2.Histo2D(
        (f"h_sumE_seed_{name}",
         f"{title}; seed ix; seed iy",
         nx, xmin, xmax,
         ny, ymin, ymax),
        "ecal_cluster_seed_ix",
        "ecal_cluster_seed_iy",
        "ecal_cluster_energy"
    )

    h_cnt = df2.Histo2D(
        (f"h_cnt_seed_{name}",
         f"{title} (counts); seed ix; seed iy",
         nx, xmin, xmax,
         ny, ymin, ymax),
        "ecal_cluster_seed_ix",
        "ecal_cluster_seed_iy",
        "clus_one"
    )

    hs = h_sumE.GetPtr()
    hc = h_cnt.GetPtr()

    h_mean = hs.Clone(f"h_meanE_seed_{name}")
    h_mean.SetTitle(f"{title} (mean cluster energy); seed ix; seed iy")
    h_mean.Divide(hc)

    return h_mean, hs, hc

# --- build maps ---
h_mean_data, h_sum_data, h_cnt_data = mean_energy_seed_map(df_data, "data", "DATA")
h_mean_mc,   h_sum_mc,   h_cnt_mc   = mean_energy_seed_map(df_mc,   "mc",   "FEE MC")

# --- draw side-by-side ---
c = R.TCanvas("c_ecal_maps", "ECAL mean energy maps", 1100, 450)
c.Divide(2,1)

c.cd(1); R.gPad.SetGrid()
# h_mean_data.Draw("COLZ")
h_sum_data.Draw("COLZ")

c.cd(2); R.gPad.SetGrid()
# h_mean_mc.Draw("COLZ")
h_sum_mc.Draw("COLZ")
c.Draw()
# c.SaveAs("plots/ecal_mean_energy_maps_data_vs_mc.png")

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_ecal_maps


In [2]:
# ------------------------------------------------------
# Create list of all seed crystals (ix, iy)
# based on the min and max in the MC sample.
# ------------------------------------------------------

# Get min and max ix, iy
ix_min = int(df_mc.Min("ecal_cluster_seed_ix").GetValue())  # -23
ix_max = int(df_mc.Max("ecal_cluster_seed_ix").GetValue())  #  23
iy_min = int(df_mc.Min("ecal_cluster_seed_iy").GetValue())  # -5
iy_max = int(df_mc.Max("ecal_cluster_seed_iy").GetValue())  #  5

# Create arrays of ix and iy values based on the min and max from MC
ix_arr = list(range(ix_min, ix_max + 1))
iy_arr = list(range(iy_min, iy_max + 1))

def is_valid_crystal(x, y):
    if y == 0: return False
    if x == 0: return False
    if -10 < x < -1 and y ==  1: return False
    if -10 < x < -1 and y == -1: return False
    if x ==  3 and y ==  5: return False
    if x == -18 and y ==  5: return False
    if x == -1  and y == -5: return False
    return True

all_pairs = list(itertools.product(ix_arr, iy_arr))
pairs = [(x, y) for (x, y) in itertools.product(ix_arr, iy_arr) if is_valid_crystal(x, y)]
excluded = [(x, y) for (x, y) in all_pairs if not is_valid_crystal(x, y)]

print(f"Total possible crystals: {len(all_pairs)}")
print(f"Total valid crystals: {len(pairs)}")
print(f"Exculded: {len(excluded)}")
print("First 10:", pairs[:10])

Total possible crystals: 517
Total valid crystals: 441
Exculded: 76
First 10: [(-23, -5), (-23, -4), (-23, -3), (-23, -2), (-23, -1), (-23, 1), (-23, 2), (-23, 3), (-23, 4), (-23, 5)]


In [ ]:
E_BEAM = 3.742  # GeV

R.gInterpreter.Declare("""
double crystalBall(double x, double norm, double mu, double sigma, double alpha, double n) {
    double t = (x - mu) / sigma;
    if (t > -alpha) {
        return norm * std::exp(-0.5 * t * t);
    } else {
        double A = std::pow(n / alpha, n) * std::exp(-0.5 * alpha * alpha);
        double B = n / alpha - alpha;
        return norm * A * std::pow(B - t, -n);
    }
}
""")
CB_FORMULA = "crystalBall(x, [0], [1], [2], [3], [4])"


def fit_fee_peak_for_seed(df, seed_ix, seed_iy, tag,
                          Emin=2.0, seedFracMin=0.6,
                          ep_window=0.3,
                          nbins=120, xlo=1.5, xhi=4.5,
                          fit_halfwidth=0.30, min_entries=20,
                          fit_type="cb", plot_dir="plots"):

    os.makedirs(plot_dir, exist_ok=True)

    d = (df
        .Filter("ecal_cluster_energy.size() > 0 && part_pdg.size() > 0")
        .Define("seed_over_etot",
                "ecal_cluster_seed_energy / (ecal_cluster_energy + 1e-9f)")
        .Define("Esel", f"""
            ROOT::VecOps::RVec<float> out;
            for (int i = 0; i < (int)part_pdg.size(); ++i) {{
                int tr = part_track[i];
                if (tr < 0) continue;
                int cl = part_ecal_cluster[i];
                if (cl < 0) continue;                                            // Make sure to have macthed track and cluster
                
                if (ecal_cluster_seed_ix[cl] != {seed_ix} ||
                    ecal_cluster_seed_iy[cl] != {seed_iy}) continue;             // Look at required crystal
                
                float eclus = ecal_cluster_energy[cl];
                if (eclus < {Emin}f) continue;                                  // Min. cluster energy check
                
                float seedfrac = seed_over_etot[cl];
                if (seedfrac < {seedFracMin}f) continue;                        // Min. seed to total energy fraction check
                
                if (part_pdg[i] != 11) continue;                                // choose only electrons
                
                float px = track_px[tr], py = track_py[tr], pz = track_pz[tr];
                float psum = std::sqrt(px*px + py*py + pz*pz);
                float ep = eclus / (psum + 1e-9f);
                if (std::abs(ep - 1.0f) > {ep_window}f) continue;               // track p and cluster energy difference check
                
                out.push_back(eclus);
            }}
            return out;
        """)
        .Filter("Esel.size() > 0")
    )

    hname = f"hE_{tag}_{seed_ix}_{seed_iy}"
    hR = d.Histo1D((hname,
                    f"Eclus seed({seed_ix},{seed_iy}), {fit_type} [{tag}]; E [GeV]; Counts",
                    nbins, xlo, xhi), "Esel")
    h = hR.GetPtr()

    peak_x = h.GetXaxis().GetBinCenter(h.GetMaximumBin())
    # peak_x = 3.7
    fname  = f"f_{tag}_{seed_ix}_{seed_iy}"

    if fit_type == "cb":
        f = R.TF1(fname, CB_FORMULA,
                  peak_x - 1.5*fit_halfwidth, peak_x + fit_halfwidth)
        f.SetParameters(h.GetMaximum(), peak_x, 0.1, 1.2, 2.0)                             # *** Changes here!!!***
        f.SetParNames("Norm", "mu", "sigma", "alpha", "n")
    else:
        f = R.TF1(fname, "gaus",
                  peak_x - fit_halfwidth, peak_x + fit_halfwidth)

    ret     = int(h.Fit(f, "RQ0"))
    n       = int(h.GetEntries())
    mu      = float(f.GetParameter(1))
    sig     = abs(float(f.GetParameter(2)))
    nDoF    = int(f.GetNDF())
    chi2_ndf = f.GetChisquare() / nDoF if nDoF > 0 else -1.0

    if n < min_entries:
        return None

    # --- per-crystal plot ---
    cname = f"c_{tag}_{seed_ix}_{seed_iy}"
    obj = R.gROOT.FindObject(cname)
    if obj: obj.Delete()

    c = R.TCanvas(cname, "", 700, 500)
    c.SetGrid()
    h.SetLineColor(R.kBlue + 1)
    h.SetLineWidth(2)
    h.SetStats(0)
    h.Draw("HIST")
    f.SetLineColor(R.kRed)
    f.SetLineWidth(2)
    f.Draw("SAME")

    lab = R.TLatex()
    lab.SetNDC()
    lab.SetTextSize(0.035)
    for y, txt in [
        (0.85, f"seed ({seed_ix}, {seed_iy})  [{tag}]"),
        (0.80, f"#mu = {mu:.4f} GeV"),
        (0.75, f"#sigma = {sig:.4f} GeV"),
        (0.70, f"#mu/E_{{beam}} = {mu/E_BEAM:.4f}"),
        (0.65, f"N = {n}"),
        (0.60, f"fit: {fit_type.upper()}"),
        (0.55, f"#chi^{{2}}/ndf = {chi2_ndf:.2f}  (nDoF={nDoF})"),
    ]:
        lab.DrawLatex(0.15, y, txt)

    c.SaveAs(os.path.join(plot_dir, f"fee_{tag}_ix{seed_ix}_iy{seed_iy}.png"))
    c.Close()

    return mu, sig, n, chi2_ndf, nDoF, ret


def make_fee_products(df, pairs, tag="MC",
                         fit_type="cb", nbins=120,
                         out_dir="fee_output", **kwargs):

    os.makedirs(out_dir, exist_ok=True)
    plot_dir = os.path.join(out_dir, f"{fit_type}_plots")
    os.makedirs(plot_dir, exist_ok=True)

    out_csv  = os.path.join(out_dir, f"fee_{tag}_peaks_{fit_type}.csv")
    out_root = os.path.join(out_dir, f"fee_{tag}_maps_{fit_type}.root")
    out_png  = os.path.join(out_dir, f"fee_{tag}_muOverEbeam_{fit_type}.png")
    out_png_n = os.path.join(out_dir, f"fee_{tag}_nhits_{fit_type}.png")

    # clean up any leftover ROOT objects from previous runs
    for name in ["h_muOverEbeam", "h_nhits", "c_fee_muOverE", "c_fee_nhits"]:
        obj = R.gROOT.FindObject(name)
        if obj: obj.Delete()

    h_muOverE = R.TH2D("h_muOverEbeam",
                       f"FEE {tag} #mu/E_{{beam}}; seed ix; seed iy",
                       47, -23.5, 23.5, 11, -5.5, 5.5)
    h_nhits   = R.TH2D("h_nhits",
                       f"FEE {tag} entries N(ix,iy); seed ix; seed iy",
                       47, -23.5, 23.5, 11, -5.5, 5.5)

    results, bad = [], []

    for ix, iy in pairs:
        r = fit_fee_peak_for_seed(df, ix, iy, tag,
                                  fit_type=fit_type,
                                  plot_dir=plot_dir,
                                  nbins=nbins,
                                  **kwargs)
        if r is None:
            bad.append((ix, iy))
            continue

        mu, sig, n, chi2_ndf, nDoF, ret = r
        results.append((ix, iy, n, mu, sig, chi2_ndf, nDoF, mu / E_BEAM, ret))

        xbin = h_muOverE.GetXaxis().FindBin(ix)
        ybin = h_muOverE.GetYaxis().FindBin(iy)
        h_muOverE.SetBinContent(xbin, ybin, mu / E_BEAM)
        h_nhits.SetBinContent(xbin, ybin, n)

    # --- CSV ---
    with open(out_csv, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["seed_ix", "seed_iy", "n", "mu_GeV", "sigma_GeV",
                    "chi2_ndf", "DoF", "mu_over_Ebeam", "fit_status"])
        w.writerows(results)

    # --- ROOT file ---
    fout = R.TFile(out_root, "RECREATE")
    h_muOverE.Write()
    h_nhits.Write()
    fout.Close()

    # --- summary label helper ---
    def draw_label(txt):
        lab = R.TLatex()
        lab.SetNDC()
        lab.SetTextSize(0.035)
        lab.DrawLatex(0.12, 0.92, txt)

    summary = f"FEE {tag}  |  fit: {fit_type.upper()}  |  nbins: {nbins}  |  {len(results)} crystals"

    # --- mu/Ebeam map ---
    c1 = R.TCanvas("c_fee_muOverE", f"FEE {tag} mu/Ebeam", 900, 450)
    c1.SetRightMargin(0.14)
    h_muOverE.SetStats(0)
    h_muOverE.Draw("COLZ")
    draw_label(summary)
    c1.Update()
    c1.SaveAs(out_png)

    # --- nhits map ---
    c2 = R.TCanvas("c_fee_nhits", f"FEE {tag} N hits", 900, 450)
    c2.SetRightMargin(0.14)
    h_nhits.SetStats(0)
    R.gStyle.SetPalette(R.kBird)
    h_nhits.Draw("COLZ")
    draw_label(summary)
    c2.Update()
    c2.SaveAs(out_png_n)

    # --- bad pairs ---
    bad_txt = os.path.join(out_dir, f"bad_{fit_type}.txt")
    with open(bad_txt, "w") as f:
        f.write(f"Total bad pairs: {len(bad)}\n")
        f.write(str(bad) + "\n")

    print(f"[OK] {len(results)} crystals fitted, {len(bad)} failed.")
    print(f"[OK] Outputs written to: {out_dir}/")

    return results, bad

seed(4,-2) | entries=18 | mu=3.3539637305978216 | sig=0.02365249222905162 | ret=1
seed(3,2) | entries=8 | mu=3.0454750939342055 | sig=0.024464847933563145 | ret=0
seed(10,2) | entries=6 | mu=2.8891979369212257 | sig=0.018104681722466914 | ret=0
seed(-2,2) | entries=15 | mu=3.55 | sig=0.10104146839508323 | ret=1
seed(-13,2) | entries=5 | mu=3.6299282459475943 | sig=0.25212682405901615 | ret=1
seed(-13,-2) | entries=4 | mu=2.818071746277138 | sig=0.24776148999932573 | ret=1
seed(-18,-2) | entries=1 | mu=3.515545419104427 | sig=0.2495673414801828 | ret=1
seed(3,-2) | entries=3 | mu=2.799234159861696 | sig=0.24823645693628832 | ret=1
seed(6,-2) | entries=10 | mu=3.1465084871336173 | sig=0.23776801668920106 | ret=1
seed(11,-2) | entries=0 | mu=0.0 | sig=0.0 | ret=-1
[OK] Wrote fee_mc_peaks.csv with 10 crystals.
[OK] Wrote fee_mc_maps.root and fee_mc_muOverEbeam.png.
[INFO] Failed/low-stat crystals: 0


Warning in <Fit>: Fit data is empty 
Info in <TCanvas::Print>: png file fee_mc_muOverEbeam.png has been created


In [ ]:
FIT_TYPE = "cb"
NBINS = 120
TAG = "data"
df = df_data
MIN_ENTRIES = 20
HALF_WIDTH = 1.0
OUT_DIR_MAIN = TAG
OUT_DIR = os.path.join(OUT_DIR_MAIN, f"{TAG}_nbins{NBINS}_fitwidth_1.5_{HALF_WIDTH}_w_trk_reqs")

# pairs_ = [(4, -2), (3, 2), (10, 2), (-2, 2), (-13, 2), (-13, -2), (-18, -2), (3, -2), (6, -2), (11, -2)]

# data/data_nbins120_fitwidth_1.5_1.0_w_trk_req
# pairs_ = [(-1,-1),(2,5),(-4,5),(7,1),(-7,2),
#             (7,-5),(-8,2),(-9,2),(-9,-5),(-10,2),
#             (10,5),(-11,1),(-11,-1),(11,2),(-11,2),(-11,5),
#             (12,-1),(-12,3),(12,-4),(13,-1),(-13,1),(-13,-1),
#             (-13,2),(-13,-5),(-13,5),
#             (14,1),(14,2),(-14,-2),(14,3),(14,4),(14,5),(-15,1),
#             (-15,3),(-16,-1),(-16,2),(-16,-2),(-17,-1),(-17,2),
#             (-17,-4),(-18,1),(-18,2),(-18,-3),
#             (-20,-2),(-21,-3),(-22,2),(-22,3),(-22,4),(-22,5)]

pairs_ = pairs

results_cb, bad_cb = make_fee_products(df,
                                      pairs_,
                                      tag=TAG,
                                      fit_type=FIT_TYPE,
                                      nbins=NBINS,
                                      out_dir=OUT_DIR,
                                      min_entries=MIN_ENTRIES,
                                      fit_halfwidth=HALF_WIDTH,
)

In [ ]:
FIT_TYPE = "cb"
NBINS = 120
TAG = "MC"
df = df_mc
MIN_ENTRIES = 20
HALF_WIDTH = 1.0
OUT_DIR_MAIN = TAG
OUT_DIR = os.path.join(OUT_DIR_MAIN, f"{TAG}_nbins{NBINS}_fitwidth_1.5_{HALF_WIDTH}_w_trk_reqs")

# pairs_ = [(4, -2), (3, 2), (10, 2), (-2, 2), (-13, 2), (-13, -2), (-18, -2), (3, -2), (6, -2), (11, -2)]
pairs_ = pairs

results_cb, bad_cb = make_fee_products(df,
                                      pairs_,
                                      tag=TAG,
                                      fit_type=FIT_TYPE,
                                      nbins=NBINS,
                                      out_dir=OUT_DIR,
                                      min_entries=MIN_ENTRIES,
                                      fit_halfwidth=HALF_WIDTH,
)